# Data Science Internship – February 2026## Task 4: Fine-Tuning BERT on a Kaggle DatasetThis notebook fine-tunes `bert-base-uncased` for text classification using Hugging Face Transformers.**Experiments included**:- Freeze BERT layers + train classification head- Fine-tune last 2 BERT encoder layers + train classification head

## Dataset (Kaggle)Default dataset: **IMDB Movie Reviews (50K)**`lakshmi25npathi/imdb-dataset-of-50k-movie-reviews`### How to provide the datasetOption A (recommended): use Kaggle API- Put `kaggle.json` in `~/.kaggle/` (or Colab upload it and move it there).Option B: download manually- Download the dataset CSV yourself- Set `DATA_PATH` in the next code cell to your CSV file path.

In [ ]:
import sysimport subprocessdef pip_install(packages):    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])# Lightweight dependencies (assumes torch is already installed).pip_install([    'transformers',    'datasets',    'kaggle',    'scikit-learn',    'seaborn',    'matplotlib'])

In [ ]:
import osimport reimport randomimport numpy as npimport pandas as pdimport torchfrom datasets import Datasetfrom transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArgumentsfrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matriximport seaborn as snsimport matplotlib.pyplot as pltSEED = 42random.seed(SEED)np.random.seed(SEED)torch.manual_seed(SEED)MODEL_NAME = 'bert-base-uncased'MAX_LENGTH = 128LR = 2e-5BATCH_SIZE = 8EPOCHS = 2# Optional speed-up (set to None for full dataset).MAX_SAMPLES = None# Kaggle dataset id (default IMDB dataset).KAGGLE_DATASET = 'lakshmi25npathi/imdb-dataset-of-50k-movie-reviews'# If Kaggle API is not available, set this to your CSV path.# Example: DATA_PATH = './IMDB Dataset.csv'DATA_PATH = None

In [ ]:
def normalize_text(s):    s = str(s)    s = s.replace('\n', ' ')    s = re.sub(r'\s+', ' ', s).strip()    return sdef load_dataset():    if DATA_PATH and os.path.exists(DATA_PATH):        return pd.read_csv(DATA_PATH)    # Kaggle API fallback.    from kaggle.api.kaggle_api_extended import KaggleApi    api = KaggleApi()    api.authenticate()    download_dir = './kaggle_download'    os.makedirs(download_dir, exist_ok=True)    api.dataset_download_files(        KAGGLE_DATASET,        path=download_dir,        unzip=True,        quiet=True,    )    csv_files = [f for f in os.listdir(download_dir) if f.lower().endswith('.csv')]    if not csv_files:        raise FileNotFoundError('No CSV file found after Kaggle download.')    return pd.read_csv(os.path.join(download_dir, csv_files[0]))# Load datadf = load_dataset()print('Loaded dataset shape:', df.shape)print('Columns:', list(df.columns))# Detect common IMDB column names (or fallback to generic text/label names).if 'review' in df.columns and 'sentiment' in df.columns:    text_col = 'review'    label_col = 'sentiment'elif 'text' in df.columns and 'label' in df.columns:    text_col = 'text'    label_col = 'label'else:    raise ValueError('Please adapt the dataset loader: set DATA_PATH to a CSV with (review, sentiment) or (text, label).')df[text_col] = df[text_col].fillna('').map(normalize_text)# Map labels to integers (IMDB: positive/negative).label_lower = df[label_col].astype(str).str.lower()if set(label_lower.unique()).issubset({'positive', 'negative'}):    df['label'] = label_lower.map({'negative': 0, 'positive': 1})else:    unique_labels = sorted(label_lower.unique().tolist())    mapping = {lab: i for i, lab in enumerate(unique_labels)}    df['label'] = label_lower.map(mapping)df = df[[text_col, 'label']].rename(columns={text_col: 'text'})if MAX_SAMPLES is not None:    df = df.sample(n=MAX_SAMPLES, random_state=SEED).reset_index(drop=True)print('Final dataset shape:', df.shape)print('Label distribution:')print(df['label'].value_counts())

In [ ]:
train_df, temp_df = train_test_split(    df,    test_size=0.2,    random_state=SEED,    stratify=df['label'],)val_df, test_df = train_test_split(    temp_df,    test_size=0.5,    random_state=SEED,    stratify=temp_df['label'],)print('Train:', train_df.shape, 'Val:', val_df.shape, 'Test:', test_df.shape)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))val_ds = Dataset.from_pandas(val_df.reset_index(drop=True))test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))def tokenize_batch(batch):    return tokenizer(        batch['text'],        padding='max_length',        truncation=True,        max_length=MAX_LENGTH,    )train_ds = train_ds.map(tokenize_batch, batched=True)val_ds = val_ds.map(tokenize_batch, batched=True)test_ds = test_ds.map(tokenize_batch, batched=True)train_ds = train_ds.rename_column('label', 'labels')val_ds = val_ds.rename_column('label', 'labels')test_ds = test_ds.rename_column('label', 'labels')train_ds = train_ds.remove_columns(['text'])val_ds = val_ds.remove_columns(['text'])test_ds = test_ds.remove_columns(['text'])train_ds.set_format(type='torch')val_ds.set_format(type='torch')test_ds.set_format(type='torch')num_labels = df['label'].nunique()print('num_labels:', num_labels)

In [ ]:
def compute_metrics(eval_pred):    logits, labels = eval_pred    preds = np.argmax(logits, axis=-1)    acc = accuracy_score(labels, preds)    if num_labels == 2:        precision, recall, f1, _ = precision_recall_fscore_support(            labels,            preds,            average='binary',            zero_division=0,        )    else:        precision, recall, f1, _ = precision_recall_fscore_support(            labels,            preds,            average='macro',            zero_division=0,        )    return {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1}def apply_experiment_settings(model, experiment_id):    # Freeze everything first.    for param in model.parameters():        param.requires_grad = False    # Unfreeze classification head.    for name, param in model.named_parameters():        if 'classifier' in name:            param.requires_grad = True    if experiment_id == 'freeze_bert':        return model    if experiment_id == 'fine_tune_last2':        # Unfreeze last 2 BERT encoder layers.        if hasattr(model, 'bert') and hasattr(model.bert, 'encoder'):            layers = model.bert.encoder.layer            for layer in layers[-2:]:                for param in layer.parameters():                    param.requires_grad = True        # Optional: unfreeze pooler too.        if hasattr(model, 'bert') and hasattr(model.bert, 'pooler'):            for param in model.bert.pooler.parameters():                param.requires_grad = True        return model    raise ValueError(f'Unknown experiment_id: {experiment_id}')def evaluate_with_confusion(trainer, dataset):    pred = trainer.predict(dataset)    logits = pred.predictions    preds = np.argmax(logits, axis=-1)    labels = pred.label_ids    metrics = compute_metrics((logits, labels))    cm = confusion_matrix(labels, preds)    return metrics, cmdef run_experiment(experiment_id, output_dir):    model = AutoModelForSequenceClassification.from_pretrained(        MODEL_NAME,        num_labels=num_labels,    )    model = apply_experiment_settings(model, experiment_id)    training_args = TrainingArguments(        output_dir=output_dir,        learning_rate=LR,        num_train_epochs=EPOCHS,        per_device_train_batch_size=BATCH_SIZE,        per_device_eval_batch_size=BATCH_SIZE,        evaluation_strategy='epoch',        save_strategy='no',        logging_strategy='epoch',        seed=SEED,        weight_decay=0.01,        fp16=torch.cuda.is_available(),        report_to=[],    )    trainer = Trainer(        model=model,        args=training_args,        train_dataset=train_ds,        eval_dataset=val_ds,        tokenizer=tokenizer,        compute_metrics=compute_metrics,    )    trainer.train()    metrics, cm = evaluate_with_confusion(trainer, test_ds)    return metrics, cmresults = {}conf_matrices = {}for exp_id, out_dir in [    ('freeze_bert', './outputs_freeze_bert'),    ('fine_tune_last2', './outputs_fine_tune_last2'),]:    print('=== Running experiment:', exp_id, '===')    metrics, cm = run_experiment(exp_id, out_dir)    results[exp_id] = metrics    conf_matrices[exp_id] = cm    print('Test metrics:', metrics)

In [ ]:
comparison = pd.DataFrame(results).Tprint('\n=== Experiment comparison (Test set) ===')print(comparison)for exp_id, cm in conf_matrices.items():    plt.figure(figsize=(5, 4))    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')    plt.title(f'Confusion Matrix - {exp_id}')    plt.xlabel('Predicted')    plt.ylabel('Actual')    plt.show()

## LinkedIn Post (copy-paste draft)**Summary:**In this NLP task, I fine-tuned BERT on a real-world Kaggle text classification dataset. I implemented preprocessing, tokenization, training experiments (freeze BERT vs fine-tune last layers), and evaluation using standard NLP metrics.**Key Learnings:**- BERT fine-tuning for sequence classification- Tokenization using `bert-base-uncased`- Training with Hugging Face Transformers (`Trainer`, `TrainingArguments`)- Evaluation using Accuracy, Precision, Recall, and F1- Confusion matrix for understanding prediction errors**Acknowledgment:**Thanks to Innomatics Research Labs, my trainer, and my mentor for the guidance and support throughout this internship.**Hashtags:** #NLP #AI #DataScience #MachineLearning